In [1]:
import os 
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import pickle
import anndata
from anndata import AnnData
import json


In [2]:
def pickle_data(data, path):

    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_pickle(path): 

    with open(path, "rb") as f: 
        return pickle.load(f)

In [3]:
cptac_folder = "/data/rbg/shared/datasets/CPTAC/CPTAC3_gene_counts"

In [9]:
def preprocess_RNA(RNA_seq_folder):

    RNA_seq_info = {} 

    for sample_id in tqdm(os.listdir(RNA_seq_folder)):

        current_folder = RNA_seq_folder / sample_id
        file_path = [f for f in os.listdir(current_folder) if "augmented_star_gene_counts.tsv" in f]
        if len(file_path) == 0: continue 
        assert len(file_path) == 1 
        file_path = file_path[0]

        expression = pd.read_csv(current_folder / file_path,  sep = "\t", skiprows=1).fillna("-")
        expression = expression[expression["gene_name"] != "-"]
        expression = {g : e for g, e in zip(expression["gene_name"].tolist(), expression["tpm_unstranded"].tolist())}

        RNA_seq_info[file_path] = expression
    
    return RNA_seq_info


In [10]:
rna_seq = preprocess_RNA(Path(cptac_folder))

100%|██████████| 2393/2393 [25:48<00:00,  1.54it/s] 


In [11]:
df = pd.DataFrame.from_dict(rna_seq, orient='index')

In [12]:
df.head()

,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,AL451106.1,AC092910.4,AC073611.1,AC136977.1,AC078856.1,AC008763.4,AL592295.6,AC006486.3,AL391628.1,AP006621.6
437aec8f-423e-47a2-b129-6b4ba114f3ee.rna_seq.augmented_star_gene_counts.tsv,28.0715,0.6563,71.5703,8.9044,8.1126,17.1586,44.9341,29.5504,27.2207,15.6430,...,0.0,17.6117,0.4172,0.0,21.6537,0.0,17.4460,0.0,0.1098,0.6290
b8cc9b0c-d4f2-4676-bf7e-b17f753cefe4.rna_seq.augmented_star_gene_counts.tsv,18.0736,0.2344,70.8597,8.4054,8.9811,30.6844,57.2733,22.9221,30.2445,33.8564,...,0.0,18.7371,0.5159,0.0,41.1129,0.0,5.1208,0.0,0.0449,0.5662
dd5b3bdf-9753-4e8a-9455-c7c51aebd459.rna_seq.augmented_star_gene_counts.tsv,13.3228,2.1408,53.0135,19.1237,9.2791,15.8542,38.3599,24.8439,54.7697,44.3851,...,0.0,41.1441,0.5371,0.0,69.6077,0.0,28.8532,0.0,0.1437,0.7053
f8d01853-9249-4385-840f-a008345120f0.rna_seq.augmented_star_gene_counts.tsv,85.9573,0.4612,90.3327,16.5240,5.1100,9.9534,77.1244,40.7413,22.9840,32.0032,...,0.0,20.6265,0.2497,0.0,29.4181,0.0,6.1297,0.0,0.0257,0.5525
58c5ea3b-47b0-45c7-a05e-b957f1de9cd1.rna_seq.augmented_star_gene_counts.tsv,44.0276,2.0531,84.0658,18.2071,7.3028,6.1854,18.6686,36.6833,28.2324,39.1165,...,0.0,32.5843,0.2744,0.0,36.8576,0.0,11.5767,0.0,0.1219,0.4655


In [13]:
cptac_adata = AnnData(X=df.values.astype(np.float32))
cptac_adata.obs_names = df.index.astype(str)
cptac_adata.var_names = df.columns.astype(str)

In [14]:
cptac_adata.obs['patient_id'] = cptac_adata.obs_names
cptac_adata.var['gene_symbol'] = cptac_adata.var_names

In [15]:
print(f"Final AnnData shape: {cptac_adata.shape}")

Final AnnData shape: (2393, 59427)


In [16]:
# Filter for genes with mean expression over 0.1
mean_expression = cptac_adata.X.mean(axis=0)
genes_to_keep = mean_expression > 0.1

# how many genes to keep
print(f"Number of genes before filtering: {cptac_adata.n_vars}")
print(f"Number of genes after filtering: {genes_to_keep.sum()}")

filtered_adata = cptac_adata[:, genes_to_keep]

Number of genes before filtering: 59427
Number of genes after filtering: 51912


### Compute log1p adata + variance

In [17]:
X_log = np.log1p(filtered_adata.X)
gene_variances = np.var(X_log, axis=0)

filtered_adata.var["log_variance"] = gene_variances

/tmp/ipykernel_2261738/1862220688.py:4: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  filtered_adata.var["log_variance"] = gene_variances


In [18]:
filtered_adata.write_h5ad("../../data/CPTAC/RNA/CPTAC_bulk_RNA_processed.h5ad")

In [27]:
filtered_adata

AnnData object with n_obs × n_vars = 2393 × 51912
    obs: 'patient_id'
    var: 'gene_symbol', 'log_variance'

In [19]:
log_adata = filtered_adata.copy()
log_adata.X = X_log
log_adata.write_h5ad("../../data/CPTAC/RNA/CPTAC_bulk_RNA_processed_log1p.h5ad")

In [20]:
example_file = pd.read_csv("/data/rbg/shared/datasets/CPTAC/CPTAC3_gene_counts/35d425e1-15c9-430e-a86f-4df0c607f575/5fc3f4ec-4e56-43bb-878b-cef48f787434.rna_seq.augmented_star_gene_counts.tsv", sep="\t", skiprows=1)

In [22]:
protein_coding_genes = example_file[example_file["gene_type"] == "protein_coding"]["gene_name"].tolist()

In [23]:
len(protein_coding_genes)

19962

In [24]:
# Get gene ordering
cptac_var = dict(zip(filtered_adata.var["gene_symbol"], filtered_adata.var["log_variance"]))

# Filter by those in selected genes
cptac_var = {gene: var for gene, var in cptac_var.items() if gene in protein_coding_genes}

# Sort the dictionary by value (variance) in descending order
selected_genes_cptac = dict(
    sorted(cptac_var.items(), key=lambda item: item[1], reverse=True)
)

In [25]:
len(selected_genes_cptac)

18775

In [26]:
pickle_data(selected_genes_cptac, "../../data/CPTAC/CPTAC_genes_ordered_by_variance.pkl")

### Get cancer type mapping

In [1]:
RNA_PATH = "../../data/CPTAC/RNA/CPTAC_bulk_RNA_processed_log1p.h5ad"
CLINICAL_PATH = "../../data/CPTAC/clinical/clinical.tsv"
METADATA_JSON = "../../data/CPTAC/clinical/metadata.cohort.all.json"

In [4]:
adata = anndata.read_h5ad(RNA_PATH)
obs_ids = adata.obs_names.astype(str).tolist()

print(f"Loaded RNA AnnData with {len(obs_ids)} samples")

Loaded RNA AnnData with 2393 samples


In [5]:
with open(METADATA_JSON, "r") as f:
    metadata = json.load(f)

file_to_case = {}

for r in metadata:
    ae = r.get("associated_entities", [])
    if not ae:
        continue

    case_id = ae[0].get("case_id", None)
    if case_id is None:
        continue

    for da in r.get("downstream_analyses", []):
        for of in da.get("output_files", []):
            if of.get("data_type") == "Gene Expression Quantification":
                fn = of.get("file_name", None)
                if fn:
                    file_to_case[fn] = case_id

print(f"Mapped {len(file_to_case)} RNA files to case_ids")

Mapped 2760 RNA files to case_ids


In [6]:
# -----------------------------
# Load clinical and build case_id -> organ mapping
# -----------------------------
clinical = pd.read_csv(CLINICAL_PATH, sep="\t")
clinical["cases.case_id"] = clinical["cases.case_id"].astype(str)

organ_cols = [
    "diagnoses.tissue_or_organ_of_origin",
    "diagnoses.primary_site",
    "diagnoses.site_of_resection_or_biopsy",
    "diagnoses.primary_diagnosis",
]

case_to_organ = {}

for _, row in clinical.iterrows():
    case_id = row["cases.case_id"]
    organ = None
    for col in organ_cols:
        if col in row.index and pd.notna(row[col]):
            organ = str(row[col]).strip()
            break
    case_to_organ[case_id] = organ

print(f"Mapped {len(case_to_organ)} case_ids to organ")

Mapped 1345 case_ids to organ


In [7]:
# -----------------------------
# Final mapping: obs_name -> organ
# -----------------------------
obs_to_organ = {}

for obs in obs_ids:
    # remove trailing .pkl to match metadata file_name
    file_name = obs[:-4] if obs.endswith(".pkl") else obs
    case_id = file_to_case.get(file_name, None)

    if case_id is None:
        obs_to_organ[obs] = None
    else:
        obs_to_organ[obs] = case_to_organ.get(case_id, None)

print("Finished building obs_to_organ mapping")

# Example
list(obs_to_organ.items())[:5]

Finished building obs_to_organ mapping


[('437aec8f-423e-47a2-b129-6b4ba114f3ee.rna_seq.augmented_star_gene_counts.tsv',
  'Brain, NOS'),
 ('b8cc9b0c-d4f2-4676-bf7e-b17f753cefe4.rna_seq.augmented_star_gene_counts.tsv',
  'Lung, NOS'),
 ('dd5b3bdf-9753-4e8a-9455-c7c51aebd459.rna_seq.augmented_star_gene_counts.tsv',
  'Lung, NOS'),
 ('f8d01853-9249-4385-840f-a008345120f0.rna_seq.augmented_star_gene_counts.tsv',
  'Uterus, NOS'),
 ('58c5ea3b-47b0-45c7-a05e-b957f1de9cd1.rna_seq.augmented_star_gene_counts.tsv',
  'Uterus, NOS')]

In [8]:
organ_counts = pd.Series(list(obs_to_organ.values())).value_counts()

In [9]:
organ_counts

Lung, NOS                                             652
Kidney, NOS                                           521
Uterus, NOS                                           493
Pancreas, NOS                                         244
Brain, NOS                                            187
Larynx, NOS                                           101
Tongue, NOS                                            48
Floor of mouth, NOS                                    37
Temporal lobe                                          19
Overlapping lesion of lip, oral cavity and pharynx     12
Oropharynx, NOS                                        10
'--                                                     9
Parietal lobe                                           9
Gum, NOS                                                8
Frontal lobe                                            7
Lip, NOS                                                7
Head, face or neck, NOS                                 5
Cheek mucosa  

In [15]:
cptac_to_tcga = {
    "Lung, NOS": "LUAD",
    "Kidney, NOS": "KIRC",
    "Uterus, NOS": "UCEC",
    "Pancreas, NOS": "PAAD",
    "Brain, NOS": "GBM",
    "Breast, NOS": "BRCA",
    "Colon, NOS": "COAD",
}

In [16]:
# print number of samples that have TCGA mapping
mapped_samples = sum(1 for organ in obs_to_organ.values() if organ in cptac_to_tcga)
mapped_samples

2100

In [17]:
# Create obs_to_tcga mapping that builds off obs_to_organl this mapping should be a dict, keys being obs
obs_to_tcga = {}
for obs, organ in obs_to_organ.items():
    if organ in cptac_to_tcga:
        obs_to_tcga[obs] = cptac_to_tcga[organ]

print(f"Mapped {len(obs_to_tcga)} samples to TCGA subtypes")
pickle_data(obs_to_tcga, "../../data/CPTAC/obs_to_tcga_mapping.pkl")

Mapped 2100 samples to TCGA subtypes


### Analyze CPTAC embeddings

In [4]:
new_cptac = load_pickle("../../data/RNA_embeddings/CPTAC_processed.pkl")

In [5]:
new_cptac_ids = list(new_cptac.keys())

In [6]:
new_cptac_ids[:5]

['096f93d8-ede7-4f5e-b427-84347170ec88',
 '6145ec3f-90c8-4fc9-8f49-9f2d699da65a',
 'c4879830-e0fe-4862-92f2-bae29f58fa47',
 '0a497fd3-c5fe-4f4b-91cf-1cc99da46d12',
 '887df04b-32fa-4f96-abe2-3fb9128ae51f']

In [7]:
tcga_dict = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/CPTAC/obs_to_tcga_mapping.pkl")

In [8]:
def load_cptac_rna_filename_to_case_id(metadata_json_path: Path):
    """
    Builds mapping:
      "<file_name>" -> "<case_id>"
    where file_name looks like:
      ffeda85e-....rna_seq.augmented_star_gene_counts.tsv
    """
    with open(metadata_json_path, "r") as f:
        meta = json.load(f)

    file_to_case: Dict[str, str] = {}
    for r in meta:
        ae = r.get("associated_entities", [])
        if not ae:
            continue
        case_id = ae[0].get("case_id", None)
        if case_id is None:
            continue

        for da in r.get("downstream_analyses", []):
            for of in da.get("output_files", []):
                if of.get("data_type") == "Gene Expression Quantification":
                    fn = of.get("file_name", None)
                    if fn:
                        file_to_case[fn] = case_id

    return file_to_case

In [9]:
file_to_case = load_cptac_rna_filename_to_case_id(Path("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/CPTAC/clinical/metadata.cohort.all.json"))

In [10]:
id_tcga_mapping = {file_to_case[k]: v for k, v in tcga_dict.items() if k in file_to_case}

In [11]:
len(id_tcga_mapping), len(tcga_dict)

(1195, 2100)

In [12]:
pickle_data(id_tcga_mapping, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/CPTAC/id_to_tcga_mapping.pkl")

In [13]:
keys_to_fill = ["organ", "cancer_type"]

n_total = len(new_cptac)
counts = {k: 0 for k in keys_to_fill}

for pid, rec in new_cptac.items():
    for k in keys_to_fill:
        v = rec.get(k, None)
        if v is None:
            rec[k] = "Unknown"
            counts[k] += 1

print("Total records:", n_total)
for k in keys_to_fill:
    print(f"Filled {k}: {counts[k]}")

Total records: 1222
Filled organ: 0
Filled cancer_type: 0


In [14]:
len(new_cptac)

1222

In [19]:
def get_feature_value_counts(embeddings, feature):
    all_values = []
    for patient_id, embedding in embeddings.items():
        val = embedding.get(feature, None)
        #if val not in [None, np.nan, "Unknown", "unknown", "'--", "NA", "na", "nan"]:
        all_values.append(val)
    return pd.Series(all_values).value_counts()

In [24]:
get_feature_value_counts(new_cptac, "organ")

Lung, NOS                                             310
Kidney, NOS                                           244
Uterus, NOS                                           227
Brain, NOS                                            176
Pancreas, NOS                                         158
Larynx, NOS                                            48
Tongue, NOS                                            20
Floor of mouth, NOS                                    18
Oropharynx, NOS                                         4
Lip, NOS                                                4
Overlapping lesion of lip, oral cavity and pharynx      4
Gum, NOS                                                3
Cheek mucosa                                            2
Head, face or neck, NOS                                 2
Tonsil, NOS                                             1
Base of tongue, NOS                                     1
Name: count, dtype: int64

In [ ]:
pickle_data(new_cptac, "../data/RNA_embeddings/CPTAC_processed_cleaned.pkl")